> **Status of this notebook.** The code below was **rewritten to fix several methodological problems of the first version** (see "What changed" at the end) and has **not been executed yet**: TensorFlow, a GPU and the dataset download were not available when it was revised, so **all code outputs are cleared**. Run it top to bottom on a GPU runtime (for example Colab) after `pip install -r requirements-deep-learning.txt`; every printed number comes from that run.

# Transfer Learning - Concepts and Overview

## 1. What is Transfer Learning?
Transfer Learning is a deep learning technique where a model trained on one task is reused (fully or partially) for a second related task. This is particularly useful when:

- The new dataset is small.
- You want faster development.
- The tasks are similar in nature (e.g., both involve natural images).

**Example:** Using a model trained on ImageNet to classify medical images.

---

## 2. How Transfer Learning Reduces Training Time

- **Shared Representations**: Early layers of deep networks learn low-level features like edges, textures, and colors. These are common across different image tasks.
- **Less Data Needed**: Pretrained models reduce dependence on large datasets.
- **Faster Convergence**: Initial weights are already close to a good solution, requiring fewer training steps.

---

## 3. Pretrained Models

Pretrained models are models trained on large datasets like ImageNet (images) or BooksCorpus (text). These models provide a great foundation for a variety of downstream tasks.

### Examples:

| Model     | Domain | Highlights                                           |
|-----------|--------|------------------------------------------------------|
| VGG       | Image  | Simple deep architecture using 3x3 convolutions     |
| ResNet    | Image  | Introduces residual (skip) connections              |
| Inception | Image  | Uses multi-scale convolutions for efficiency        |
| EfficientNet | Image | Scales depth, width, and resolution uniformly     |
| BERT      | Text   | Pretrained bidirectional transformer for NLP        |
| GPT       | Text   | Causal transformer pretrained using language modeling |

---

## 4. Fine-Tuning vs. Feature Extraction

| Aspect         | Fine-Tuning                                  | Feature Extraction                            |
|----------------|-----------------------------------------------|------------------------------------------------|
| Layers Trained | All or some layers + new classifier           | Only the new classifier layer                 |
| Flexibility    | High – can adapt to complex tasks             | Low – assumes pretrained features are enough  |
| Training Time  | More training time                            | Less training time                            |
| Use Case       | Large or complex dataset, task-specific       | Small dataset, generic tasks                  |

---

## Summary
Transfer learning saves time and resources while maintaining high performance, especially on smaller datasets. It is now a default choice in many computer vision and NLP workflows.


# Case Study 1: Feature Extraction using Pretrained VGG16

We classify cats vs dogs with a pretrained **VGG16** as a frozen feature extractor and a small dense classifier on top.

## Dataset
`cats_and_dogs_filtered` (2,000 training and 1,000 validation images, balanced 50/50), downloaded from Google's ML-education bucket.

## Data splits (important)
The download provides only `train` and `validation`. If the validation set were used both to choose between models/epochs *and* to report the final accuracy, the reported number would be optimistically biased. The 1,000 validation images are therefore split deterministically into **500 validation** (model selection, early stopping) and **500 test** (reported once, for the selected model).

## Steps
- Load and split the data
- Use the **model-specific `preprocess_input`** (VGG16 and ResNet50 do not expect 0-1 scaling)
- Train a dense head on the frozen VGG16 base
- Compare feature extraction with two-stage ResNet50 fine-tuning
- Evaluate the selected model once on the test set

![VGG16](https://storage.googleapis.com/lds-media/images/vgg16-architecture.width-1200.jpg)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import VGG16, ResNet50
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg_preprocess
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
keras.utils.set_random_seed(SEED)
IMG_SIZE = (160, 160)
BATCH = 32

In [ ]:
# Download and locate the dataset
zip_path = keras.utils.get_file(
    fname='cats_and_dogs_filtered.zip',
    origin='https://storage.googleapis.com/mledu-datasets/cats_and_dogs_filtered.zip',
    extract=True,
)
base = os.path.splitext(zip_path)[0]
candidates = [os.path.join(base + '_extracted', 'cats_and_dogs_filtered'), os.path.join(base, 'cats_and_dogs_filtered'), base]
data_dir = next(d for d in candidates if os.path.isdir(os.path.join(d, 'train')))
train_dir, holdout_dir = os.path.join(data_dir, 'train'), os.path.join(data_dir, 'validation')

train_ds = keras.utils.image_dataset_from_directory(train_dir, image_size=IMG_SIZE, batch_size=BATCH, seed=SEED)
# Split the provided validation folder 50/50 into validation and test, with a fixed seed so the split is reproducible
val_ds, test_ds = keras.utils.image_dataset_from_directory(
    holdout_dir, image_size=IMG_SIZE, batch_size=BATCH, validation_split=0.5, subset='both', seed=SEED, shuffle=False)
class_names = train_ds.class_names

def class_counts(ds):
    y = np.concatenate([l.numpy() for _, l in ds])
    return dict(zip(class_names, np.bincount(y, minlength=2).tolist()))
print("classes:", class_names)
print("train:", class_counts(train_ds), "| val:", class_counts(val_ds), "| test:", class_counts(test_ds))

AUTOTUNE = tf.data.AUTOTUNE
train_ds, val_ds, test_ds = (d.cache().prefetch(AUTOTUNE) for d in (train_ds, val_ds, test_ds))

`shuffle=False` with `subset='both'` returns two disjoint, reproducible halves of the folder. The class counts printed above should be close to 250 cats / 250 dogs in each; if one half were dominated by a single class, accuracy on it would be misleading, which is why the counts are checked.

### Model builder

Images stay as raw 0-255 pixels in the datasets (easy to display). The **model-specific preprocessing is a layer inside the model**, and the pretrained base is called with `training=False`, which keeps any batch-normalisation layers in inference mode (essential for ResNet50, harmless for VGG16). Light random flips are applied only during training. Every experiment builds a **fresh** pretrained base, so one run cannot contaminate the next.

In [ ]:
def build_model(app, preprocess, dense_units=128, dropout=0.0):
    """Fresh pretrained base (frozen) + small head. Returns (model, base)."""
    base = app(input_shape=IMG_SIZE + (3,), include_top=False, weights='imagenet')
    base.trainable = False
    inputs = keras.Input(shape=IMG_SIZE + (3,))
    x = layers.RandomFlip('horizontal')(inputs)           # active only during training
    x = layers.Lambda(preprocess)(x)                      # model-specific preprocessing (not Rescaling(1/255))
    x = base(x, training=False)                           # keep BatchNorm in inference mode
    x = layers.GlobalAveragePooling2D()(x)
    if dropout: x = layers.Dropout(dropout)(x)
    x = layers.Dense(dense_units, activation='relu')(x)
    if dropout: x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    return keras.Model(inputs, outputs), base

def compile_model(model, lr):
    model.compile(optimizer=keras.optimizers.Adam(lr), loss='binary_crossentropy', metrics=['accuracy'])

early_stop = lambda: keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
results = {}   # name -> validation accuracy (used to choose the final model)

## Experiment A: VGG16 as a frozen feature extractor

Only the dense head is trained (learning rate 1e-3). The test set is not touched.

In [ ]:
vgg_model, vgg_base = build_model(VGG16, vgg_preprocess)
compile_model(vgg_model, 1e-3)
hist_vgg = vgg_model.fit(train_ds, validation_data=val_ds, epochs=10, callbacks=[early_stop()])
results['A: VGG16 frozen'] = vgg_model.evaluate(val_ds, verbose=0)[1]
print(f"Validation accuracy: {results['A: VGG16 frozen']:.4f}")

## Evaluation & Visualization (validation set)

Correct and incorrect predictions are shown from the **validation** set, which may be inspected while developing; the test set stays unseen.

In [ ]:
def collect(model, ds, n=64):
    imgs, labels = next(iter(ds.unbatch().batch(n)))
    probs = model.predict(imgs, verbose=0).ravel()
    return imgs, labels.numpy().astype(int), (probs > 0.5).astype(int)

def show_images(images, labels, preds, correct=True, max_images=6):
    plt.figure(figsize=(15, 8))
    shown = 0
    for i in range(len(images)):
        if (preds[i] == labels[i]) == correct:
            plt.subplot(2, 3, shown + 1)
            plt.imshow(images[i].numpy().astype('uint8'))      # raw 0-255 pixels, no un-normalising needed
            plt.title(f"True: {class_names[labels[i]]}\nPred: {class_names[preds[i]]}", color='green' if correct else 'red')
            plt.axis('off'); shown += 1
        if shown == max_images: break
    plt.suptitle("Correct Predictions" if correct else "Incorrect Predictions", fontsize=16)
    plt.tight_layout(); plt.show()

In [ ]:
imgs, labs, preds = collect(vgg_model, val_ds)
show_images(imgs, labs, preds, correct=True)
show_images(imgs, labs, preds, correct=False)

# Case Study 2: Fine-Tuning with ResNet50

We now unfreeze the top layers of a pretrained ResNet50 and fine-tune it for our dataset.

## Scenario
You already did feature extraction, but want better performance or you're working with a more complex dataset.

## Steps:
- Load pretrained model
- Freeze base, then unfreeze some top layers
- Retrain with a low learning rate

![ResNet50](https://towardsdatascience.com/wp-content/uploads/2022/08/0tH9evuOFqk8F41FG.png)

### Experiment B: ResNet50, two-stage fine-tuning

The standard recipe: (1) train only the new head with the base frozen, so the random head does not send large gradients into the pretrained layers; (2) unfreeze the top part of the base and continue with a **very low learning rate**. Batch-normalisation layers stay frozen throughout, and the base keeps being called with `training=False`.

In [ ]:
res_model, res_base = build_model(ResNet50, resnet_preprocess)

# Stage 1: head only
compile_model(res_model, 1e-3)
hist_res1 = res_model.fit(train_ds, validation_data=val_ds, epochs=5, callbacks=[early_stop()])

# Stage 2: unfreeze the top 50 % of the base (BatchNorm layers stay frozen), very low learning rate
res_base.trainable = True
fine_tune_at = len(res_base.layers) // 2
for layer in res_base.layers[:fine_tune_at]:
    layer.trainable = False
for layer in res_base.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False
compile_model(res_model, 1e-5)           # must re-compile after changing trainable flags
hist_res2 = res_model.fit(train_ds, validation_data=val_ds, epochs=10, callbacks=[early_stop()])
results['B: ResNet50 fine-tuned'] = res_model.evaluate(val_ds, verbose=0)[1]
print(f"Validation accuracy: {results['B: ResNet50 fine-tuned']:.4f}")

In [ ]:
acc = hist_res1.history['accuracy'] + hist_res2.history['accuracy']
val_acc = hist_res1.history['val_accuracy'] + hist_res2.history['val_accuracy']
plt.plot(acc, label='train acc'); plt.plot(val_acc, label='val acc')
plt.axvline(len(hist_res1.history['accuracy']) - 0.5, color='gray', linestyle=':', label='start of fine-tuning')
plt.title('ResNet50: accuracy over epochs'); plt.xlabel('epoch'); plt.legend(); plt.show()

## Experiment C: tackling over-fitting

Training accuracy far above validation accuracy signals over-fitting. Here a **fresh** ResNet50 (not the already fine-tuned model from Experiment B, as in the first version of this notebook) gets dropout in the head and fewer unfrozen layers (the top 10), trained with the same two-stage recipe.

In [ ]:
reg_model, reg_base = build_model(ResNet50, resnet_preprocess, dropout=0.3)
compile_model(reg_model, 1e-3)
hist_reg1 = reg_model.fit(train_ds, validation_data=val_ds, epochs=5, callbacks=[early_stop()])

reg_base.trainable = True
for layer in reg_base.layers[:-10]:
    layer.trainable = False
for layer in reg_base.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False
compile_model(reg_model, 1e-5)
hist_reg2 = reg_model.fit(train_ds, validation_data=val_ds, epochs=20, callbacks=[early_stop()])
results['C: ResNet50 + dropout, top-10 unfrozen'] = reg_model.evaluate(val_ds, verbose=0)[1]
print(f"Validation accuracy: {results['C: ResNet50 + dropout, top-10 unfrozen']:.4f}")

## Model selection and the single test evaluation

The experiments are compared on **validation** accuracy only. The winner is then evaluated **once** on the test set. With 500 test images, one image is 0.2 percentage points and the 95 % confidence interval of an accuracy near 95 % is roughly ±2 points, so differences between experiments of 1-2 points are not decisive.

In [ ]:
import pandas as pd
summary = pd.Series(results, name='validation accuracy').sort_values(ascending=False)
display(summary.round(4).to_frame())

models = {'A: VGG16 frozen': vgg_model, 'B: ResNet50 fine-tuned': res_model, 'C: ResNet50 + dropout, top-10 unfrozen': reg_model}
best_name = summary.index[0]
best_model = models[best_name]

y_true = np.concatenate([l.numpy() for _, l in test_ds]).astype(int)
y_prob = best_model.predict(test_ds, verbose=0).ravel()
y_pred = (y_prob > 0.5).astype(int)
n = len(y_true); test_acc = (y_pred == y_true).mean()
half_width = 1.96 * np.sqrt(test_acc * (1 - test_acc) / n)
print(f"Selected on validation: {best_name}")
print(f"TEST accuracy (500 images, evaluated once): {test_acc:.4f}  (95% CI approx. ±{half_width:.3f}; majority-class baseline = 0.50)")
print(classification_report(y_true, y_pred, target_names=class_names, digits=3))
print(confusion_matrix(y_true, y_pred))

## What changed from the first version (and why)

| Problem in the first version | Fix |
|---|---|
| Images scaled with `Rescaling(1/255)` although VGG16/ResNet50 expect their own `preprocess_input` | Model-specific `preprocess_input` layer inside each model |
| The validation set was used to compare models/epochs **and** to report final accuracy | Validation folder split into 500 validation + 500 test; test used once for the selected model |
| ResNet50 fine-tuned from epoch 1 with a randomly initialised head and unfrozen batch-norm layers | Two-stage training; batch-norm kept frozen; base called with `training=False` |
| The "overfitting" experiment re-used `base_model` that had already been fine-tuned, with changed trainable flags | A fresh pretrained base for every experiment |
| Fixed epoch counts, no early stopping | `EarlyStopping` on validation loss with best weights restored |
| Hard-coded `/root/.keras/...` path and a markdown claim of `cats_vs_dogs` data | Path derived from the download; dataset named correctly |
| No class-balance check, no baseline, no uncertainty on the final number | Class counts printed, 50 % baseline stated, confidence interval reported |

**Remaining limitations.** One training run per experiment (no repeated seeds); 2,000 training images from a clean, curated dataset; results do not predict accuracy on photos from other sources.